In [1]:
import os
import glob
import pandas as pd
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = r"C:\Users\AnneB\Downloads\Alzheimer MRI Detection\src\Data"

In [6]:
import os
DATA_DIR = "/Users/abechard/Alzheimer MRI Detection/Alzheimer-MRI-Detection/src/Data"  # à ajuster si le sous-dossier Data est ailleurs
print(os.listdir(DATA_DIR))

# puis un cran plus profond sur le premier sujet trouvé
first = os.listdir(DATA_DIR)[0]
print(os.listdir(os.path.join(DATA_DIR, first)))

['OAS2_RAW_PART1', 'OAS2_RAW_PART2', 'oasis_longitudinal_demographics-8d83e569fa2e2d30.xlsx']
['OAS2_0071_MR2', 'OAS2_0032_MR1', 'OAS2_0026_MR1', 'OAS2_0067_MR3', 'OAS2_0073_MR3', 'OAS2_0073_MR4', 'OAS2_0067_MR4', 'OAS2_0027_MR1', 'OAS2_0070_MR5', 'OAS2_0058_MR2', 'OAS2_0070_MR2', 'OAS2_0064_MR2', 'OAS2_0099_MR2', 'OAS2_0066_MR2', 'OAS2_0031_MR1', 'OAS2_0070_MR3', 'OAS2_0064_MR3', 'OAS2_0058_MR3', 'OAS2_0070_MR4', 'OAS2_0030_MR1', 'OAS2_0018_MR1', 'OAS2_0073_MR5', 'OAS2_0067_MR2', 'OAS2_0073_MR2', 'OAS2_0098_MR2', 'OAS2_0088_MR2', 'OAS2_0077_MR2', 'OAS2_0063_MR2', 'OAS2_0020_MR1', 'OAS2_0034_MR1', 'OAS2_0008_MR1', 'OAS2_0048_MR4', 'OAS2_0048_MR3', 'OAS2_0049_MR3', 'OAS2_0061_MR3', 'OAS2_0009_MR1', 'OAS2_0035_MR1', 'OAS2_0021_MR1', 'OAS2_0062_MR2', 'OAS2_0076_MR2', 'OAS2_0048_MR2', 'OAS2_0060_MR2', 'OAS2_0048_MR5', 'OAS2_0037_MR1', 'OAS2_0023_MR1', 'OAS2_0062_MR3', 'OAS2_0076_MR3', 'OAS2_0089_MR3', 'OAS2_0022_MR1', 'OAS2_0036_MR1', 'OAS2_0075_MR2', 'OAS2_0061_MR2', 'OAS2_0049_MR2', 'OAS

In [19]:
def print_tree(path, prefix="", max_depth=4, depth=0):
    """Affiche l'arborescence complète d'un dossier, pour diagnostic visuel."""
    if depth > max_depth or not os.path.isdir(path):
        return
    for entry in sorted(os.listdir(path)):
        full = os.path.join(path, entry)
        print(prefix + entry)
        if os.path.isdir(full):
            print_tree(full, prefix + "  ", max_depth, depth + 1)

In [20]:
part1_dir = os.path.join(DATA_DIR, "OAS2_RAW_PART1")
example_subject_dir = os.path.join(part1_dir, sorted(os.listdir(part1_dir))[0])
print(f"\n--- Arborescence de {os.path.basename(example_subject_dir)} ---")
print_tree(example_subject_dir)
 


--- Arborescence de OAS2_0001_MR1 ---
RAW
  mpr-1.nifti.hdr
  mpr-1.nifti.img
  mpr-2.nifti.hdr
  mpr-2.nifti.img
  mpr-3.nifti.hdr
  mpr-3.nifti.img


In [21]:
def inventory_subjects(data_dir=DATA_DIR):
    """Trouve tous les sujets OAS2_*_MR* dans PART1 et PART2."""
    subjects = []
    pattern = os.path.join(data_dir, "OAS2_RAW_PART*", "OAS2_*_MR*")
    for subj_path in sorted(glob.glob(pattern)):
        subj_id = os.path.basename(subj_path)
        # recherche récursive de tout fichier image, peu importe le sous-dossier
        img_files = glob.glob(os.path.join(subj_path, "**", "*.img"), recursive=True) + \
                    glob.glob(os.path.join(subj_path, "**", "*.nii*"), recursive=True)
        seg_files = glob.glob(os.path.join(subj_path, "**", "*fseg*"), recursive=True) + \
                    glob.glob(os.path.join(subj_path, "**", "FSL_SEG", "*"), recursive=True)
        subjects.append({
            "subject_id": subj_id,
            "path": subj_path,
            "n_image_files": len(img_files),
            "has_any_volume": len(img_files) > 0,
            "has_seg": len(seg_files) > 0,
        })
    return pd.DataFrame(subjects)
 
 
inv = inventory_subjects()
print(f"\n{len(inv)} sujets trouvés.")
print(inv.head(10))
print("\nSujets avec au moins un fichier image :", inv["has_any_volume"].sum())
print("Sujets avec une segmentation détectée :", inv["has_seg"].sum())
 


373 sujets trouvés.
      subject_id                                               path  \
0  OAS2_0001_MR1  /Users/abechard/Alzheimer MRI Detection/Alzhei...   
1  OAS2_0001_MR2  /Users/abechard/Alzheimer MRI Detection/Alzhei...   
2  OAS2_0002_MR1  /Users/abechard/Alzheimer MRI Detection/Alzhei...   
3  OAS2_0002_MR2  /Users/abechard/Alzheimer MRI Detection/Alzhei...   
4  OAS2_0002_MR3  /Users/abechard/Alzheimer MRI Detection/Alzhei...   
5  OAS2_0004_MR1  /Users/abechard/Alzheimer MRI Detection/Alzhei...   
6  OAS2_0004_MR2  /Users/abechard/Alzheimer MRI Detection/Alzhei...   
7  OAS2_0005_MR1  /Users/abechard/Alzheimer MRI Detection/Alzhei...   
8  OAS2_0005_MR2  /Users/abechard/Alzheimer MRI Detection/Alzhei...   
9  OAS2_0005_MR3  /Users/abechard/Alzheimer MRI Detection/Alzhei...   

   n_image_files  has_any_volume  has_seg  
0              3            True    False  
1              3            True    False  
2              4            True    False  
3              4     

In [22]:

def find_clinical_file(data_dir=DATA_DIR):
    candidates = glob.glob(os.path.join(data_dir, "*longitudinal*.xlsx")) + \
                 glob.glob(os.path.join(data_dir, "*longitudinal*.csv"))
    return candidates[0] if candidates else None
 
 
clinical_path = find_clinical_file()
print("\nFichier clinique trouvé :", clinical_path)
if clinical_path:
    clinical = pd.read_excel(clinical_path) if clinical_path.endswith(".xlsx") else pd.read_csv(clinical_path)
    print(clinical.columns.tolist())
    print(clinical.head())


Fichier clinique trouvé : /Users/abechard/Alzheimer MRI Detection/Alzheimer-MRI-Detection/src/Data/oasis_longitudinal_demographics-8d83e569fa2e2d30.xlsx
['Subject ID', 'MRI ID', 'Group', 'Visit', 'MR Delay', 'M/F', 'Hand', 'Age', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF']
  Subject ID         MRI ID        Group  Visit  MR Delay M/F Hand  Age  EDUC  \
0  OAS2_0001  OAS2_0001_MR1  Nondemented      1         0   M    R   87    14   
1  OAS2_0001  OAS2_0001_MR2  Nondemented      2       457   M    R   88    14   
2  OAS2_0002  OAS2_0002_MR1     Demented      1         0   M    R   75    12   
3  OAS2_0002  OAS2_0002_MR2     Demented      2       560   M    R   76    12   
4  OAS2_0002  OAS2_0002_MR3     Demented      3      1895   M    R   80    12   

   SES  MMSE  CDR         eTIV      nWBV       ASF  
0  2.0  27.0  0.0  1986.550000  0.696106  0.883440  
1  2.0  30.0  0.0  2004.479526  0.681062  0.875539  
2  NaN  23.0  0.5  1678.290000  0.736336  1.045710  
3  NaN  28.0  0.5

In [23]:

inv.to_csv("inventaire_oasis2.csv", index=False)
print("\nSauvegardé : inventaire_oasis2.csv")


Sauvegardé : inventaire_oasis2.csv
